In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Step 1: Create the original dataframe
df = pd.read_excel("Magic quadtrant 112625 VJay.xlsx")

df



ModuleNotFoundError: No module named 'pandas'

In [ ]:
df["Opportunity Value"] = (df["Total Contract Value"] + df["Strategic Value "] + df["Customer Intimacy"]) / 3
df["Delivery Confidence"] =(df["GDIT  Capability "] + df["Solution Complexity"] + 5 - df["Implementation Risk"]) / 3
df = df.round(2)
df

,PID,Project,Total Contract Value,Strategic Value,Customer Intimacy,GDIT Capability,Solution Complexity,Implementation Risk,Opportunity Value Score,Delivery Confidence Score,FIT Score,Effort Category,Pursue (G/Y/R),Opportunity Value,Delivery Confidence
0,P1,TREAS TCSC,2,5,3,5,5,3,NaN,NaN,NaN,NaN,NaN,3.33,4.00
1,P2,IRS IRWorks,1,3,5,5,3,1,NaN,NaN,NaN,NaN,NaN,3.00,4.00
2,P3,TREAS TFS,5,4,2,4,4,4,NaN,NaN,NaN,NaN,NaN,3.67,3.00
3,P4,TREAS FinCEn,5,2,1,2,3,5,NaN,NaN,NaN,NaN,NaN,2.67,1.67
4,P5,IRS Mainframe,2,4,4,4,3,3,NaN,NaN,NaN,NaN,NaN,3.33,3.00
5,P6,IRS PRPS,2,3,4,5,2,3,NaN,NaN,NaN,NaN,NaN,3.00,3.00
6,P7,TREAS OCC EDM,2,2,1,4,3,3,NaN,NaN,NaN,NaN,NaN,1.67,3.00
7,P8,IRS Live Assistance,2,3,4,4,4,3,NaN,NaN,NaN,NaN,NaN,3.00,3.33
8,P9,IRS CFAM,2,5,5,5,4,2,NaN,NaN,NaN,NaN,NaN,4.00,4.00
9,P10,IRS CIG,4,5,4,3,5,5,NaN,NaN,NaN,NaN,NaN,4.33,2.67


In [ ]:
# Step 2: Consolidate projects with same Metric A & Metric B
df_grouped = (
    df.groupby(["Opportunity Value", "Delivery Confidence"])["PID"]
      .apply(lambda names: "&".join(names))  # concatenate names
      .reset_index()
)

df_grouped

,Opportunity Value,Delivery Confidence,PID
0,1.67,3.00,P7
1,2.67,1.67,P4
2,3.00,3.00,P6
3,3.00,3.33,P8&P15
4,3.00,4.00,P2
5,3.33,2.67,P22
6,3.33,3.00,P5&P19
7,3.33,3.67,P14
8,3.33,4.00,P1
9,3.67,3.00,P3&P11


In [ ]:

def quadrant(df):

    # Step 3: Create Magic Quadrant chart
    fig = px.scatter(
        df,
        x="Delivery Confidence",
        y="Opportunity Value",
        text="PID",
        width=800,
        height=800
    )

    # Add quadrant lines at midpoint (3)
    fig.add_shape(type="line", x0=3, y0=1, x1=3, y1=5, line=dict(color="gray", dash="dash"))
    fig.add_shape(type="line", x0=1, y0=3, x1=5, y1=3, line=dict(color="gray", dash="dash"))

    fig.add_shape(type="line", x0=1, y0=1, x1=1, y1=5, line=dict(color="black"))
    fig.add_shape(type="line", x0=1, y0=1, x1=5, y1=1, line=dict(color="black"))
    fig.add_shape(type="line", x0=5, y0=1, x1=5, y1=5, line=dict(color="black"))
    fig.add_shape(type="line", x0=1, y0=5, x1=5, y1=5, line=dict(color="black"))

    fig.add_annotation(x=4, y=5, xanchor="center", yanchor="top", text="Pursue", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=2, y=5, xanchor="center", yanchor="top",  text="Sub/Partnership", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=2, y=1, xanchor="center", yanchor="bottom", text="Sunset", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=4, y=1, xanchor="center", yanchor="bottom", text="Low ROI", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))

    # Adjust text labels
    fig.update_traces(textposition="top center")

 
    return fig

magic_q = quadrant(df_grouped)

magic_q.show()


In [ ]:

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.025,
    column_widths=[0.75, 0.25],
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=(None, None)
)

# Left subplot (scatter)
for trace in magic_q.data:
    fig.add_trace(trace, row=1, col=1)

# Copy annotations
if hasattr(magic_q.layout, "annotations"):
    fig.update_layout(annotations=magic_q.layout.annotations)

# Copy shapes
if hasattr(magic_q.layout, "shapes"):
    fig.update_layout(shapes=magic_q.layout.shapes)


# Right subplot (table)
table = go.Table(
    columnwidth=[1,4],
    header=dict(values=["PID","Project"], fill_color="lightgray", align="left"),
    cells=dict(values=[df["PID"], df["Project"]],  align="left", height=26.5)
)

fig.add_trace(table, row=1, col=2)

fig.update_layout(
    xaxis=dict(dtick=1, title="Delivery Confidence"),
    yaxis=dict(dtick=1, title="Opportunity Value"),
    width=1200, 
    height=800,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=16, color="black", family="Helvetica Bold")
)


fig.show()


In [ ]:
df.to_excel("Output.xlsx", index=False)